# Let's try some machine learning!

In [ ]:
import src.machine_learning as ML 
# Use original PNG size
dataset = ML.GlyphDataset('data/stars-large.zip')
dataset.show()

# Resize to 64x64 and show only 3 images
dataset_resized = ML.GlyphDataset('data/stars-and-letters.zip', resize=(64, 64))
dataset_resized.show(3)

# Added exceptions
dataset = ML.GlyphDataset('data/simple-star.zip')
dataset.show(51)

## Create the DataLoader 

In [ ]:
import src.machine_learning as ML 
# 1. Create your dataset
dataset = ML.GlyphDataset('data/stars-large.zip')

# 2. Create DataLoader
loader1 = ML.create_loader(
    dataset,
    batch_size=32,
    shuffle=True,
    num_workers=0
)

#Creating loader 2 for same dataset
loader2 = ML.create_loader(
    dataset,
    batch_size=64,
    shuffle=True,
    num_workers=0,
)

# 3. Visualize the loaders
ML.visualize_loader(loader1)
ML.visualize_loader(loader2, max_images=10, nrow=5, silent=True)


## Prepare dataset for CNN

In [ ]:
import mglyph as mg
import numpy as np
import random
from typing import Callable
import src.dataset_forming as df

def star(x: float, canvas: mg.Canvas, bordercolor, fillcolor, linewidth) -> None:
    canvas.tr.translate(0, mg.lerp(x, 0, 0.05))
    radius = mg.lerp(x, 0.01, canvas.ysize / 2)

    vertices = []
    for segment in range(5):
        vertices.append(mg.orbit(canvas.center, segment * 2 * np.pi / 5, radius))
        vertices.append(mg.orbit(canvas.center, (segment + 0.5) * 2 * np.pi / 5,
                         np.cos(2 * np.pi / 5) / np.cos(np.pi / 5) * radius))

    canvas.polygon(vertices, linecap='round', style='fill', color=fillcolor) 
    canvas.polygon(vertices, width=linewidth, linecap='round', style='stroke', color=bordercolor) 

def simple_star() -> Callable[[float, mg.Canvas], None]:
    bordercolor = 'navy'
    fillcolor = 'white'
    linewidth = '70p'  
    return lambda x, canvas: star(x, canvas, bordercolor, fillcolor, linewidth)
def random_star() -> Callable[[float, mg.Canvas], None]:
    bordercolor = df.random_color()
    fillcolor = random.choice([df.random_color(), 'white'])
    linewidth = df.random_width(min=3, max=150)
    return lambda x, canvas: star(x, canvas, bordercolor, fillcolor, linewidth)
    

In [ ]:
# Simple Star for training
with df.GlyphExporter(filepath='data/simple-star.zip', dataset_name="Star (simple)") as exporter:
    shit_to_disappear = {"version": "1.0.0", 'name': ""}
    for _ in range(10):
        exporter.add(mg.export(simple_star(), path=None, short_name="star", silent=True, **shit_to_disappear,
                               xvalues=np.random.uniform(0.0, 100.0, 50)), split='train')
# Simple Star for testing 
with df.GlyphExporter(filepath='data/simple-star-test.zip', dataset_name="Star (simple-test)") as exporter:
    shit_to_disappear = {"version": "1.0.0", 'name': ""}
    for _ in range(5):
        exporter.add(mg.export(simple_star(), path=None, short_name="star", silent=True, **shit_to_disappear,
                               xvalues=np.random.uniform(0.0, 100.0, 10)), split='test')
# Random Stars for testing 
with df.GlyphExporter(filepath='data/stars-small-test.zip', dataset_name="Stars (small)") as exporter:
    shit_to_disappear = {"version": "1.0.0", 'name': ""}
    for _ in range(10):
        exporter.add(mg.export(random_star(), path=None, short_name="star", silent=True, **shit_to_disappear,
                               xvalues=np.random.uniform(0.0, 100.0, 10)), split='test')

## Create the Neural Network

In [ ]:
import src.machine_learning as ML

dataset = ML.GlyphDataset('data/simple-star.zip')
dataset.show()

In [ ]:

ML.train("data/simple-star.zip")          # Train on zipped dataset
ML.disptrain()                            # Plot training loss
ML.test("data/simple-star-test.zip")      # Evaluate on a test zip
ML.disptest()                             # Show test predictions

Testing on **Random stars** set

In [ ]:
ML.test("data/stars-small-test.zip")
ML.disptest()

In [ ]:

#  resnet-50, pre-trained on ImageNet
